# [수학 Retreat #6] 입체의 아름다움: 3D 정다면체와 가상 회전체 모델러

이 주피터 노트북은 중학교 1학년 과정의 **입체도형의 성질** 단원을 파이썬 3D 그래픽을 통해 시각적·입체적으로 탐구하는 실습 교재입니다.
기하학의 아름다운 두 축인 **'대칭성(플라톤 정다면체)'**과 **'연속성(회전체)'**을 다룹니다.

**⚠️ 학습 안내:**
본 노트북은 파이썬을 처음 접하시는 분들도 코드의 작동 원리를 명확하게 이해할 수 있도록 **모든 로직을 세포 단위 셀로 쪼개어** 구성했으며, 모든 코드 라인에 쉬운 주석을 덧붙였습니다.
각 셀을 순서대로 실행(`Shift + Enter`)하면서 수학적 정리가 화면 위 3차원 공간 속에서 구현되는 과정을 직접 확인해 보세요.


## 1. 환경 준비 및 필수 라이브러리 설치
본 실습에서는 3차원 동적 그래픽을 위해 `plotly`, 기하학적 볼록 껍질 연산을 위해 `scipy`, 그리고 Jupyter 상의 인터랙티브 제어 위젯을 만들기 위해 `ipywidgets`를 활용합니다.
아래 셀을 실행하여 필요한 라이브러리를 먼저 준비합니다.


In [ ]:
# !pip install 명령어는 주피터 노트북 내에서 터미널의 패키지 매니저를 호출하여 패키지를 설치하게 하는 매직 명령어입니다.
# -q (quiet) 옵션은 설치 시 화면에 복잡한 설치 메시지를 생략하고 깔끔하게 완료하도록 돕습니다.
!pip install -q numpy scipy plotly ipywidgets


### 1.2 모듈 임포트 (Import)
기하 연산과 3D 렌더링에 사용되는 수학 및 시각화 도구들을 파이썬 메모리에 로드합니다.


In [ ]:
# numpy: 수치 연산과 고성능 다차원 배열(좌표 격자 등)을 처리하기 위한 파이썬의 표준 수학 라이브러리입니다.
import numpy as np

# plotly.graph_objects: 마우스로 드래그하여 실시간 회전, 휠 조절을 할 수 있는 고품질 3D 그래픽 궤적을 만드는 모듈입니다.
import plotly.graph_objects as go

# scipy.spatial.ConvexHull: 임의의 3차원 좌표점들로부터 겉표면(볼록 다면체의 껍질)을 자동으로 수학 계산해주는 고급 공간 연산 모듈입니다.
from scipy.spatial import ConvexHull

# ipywidgets: 화면 상에 드롭다운 메뉴, 제어 슬라이더 등 UI 위젯을 띄워 파이썬 함수와 바인딩해주는 도구입니다.
import ipywidgets as widgets

# IPython.display: 대시보드 리포트를 위해 스타일링된 예쁜 HTML 박스를 주피터 내에 출력해 주는 라이브러리입니다.
from IPython.display import display, HTML


---
## 2. 실습 1: 5대 플라톤 정다면체와 오일러 정리
모든 면이 합동인 정다각형이고, 각 꼭짓점에 모이는 면의 개수가 모두 같은 볼록 입체도형을 **정다면체**라고 하며, 3차원 공간 속에서는 오직 5가지(**정사면체, 정육면체, 정팔면체, 정십이면체, 정이십면체**)만이 수학적으로 허락됩니다.

먼저 각 정다면체의 꼭짓점 3D 좌표($(x, y, z)$)를 수학 공식에 맞추어 생성하는 함수를 설계합니다.
이때, 5각형 대칭을 가진 정십이면체와 정이십면체의 꼭짓점 비율은 황금비 $\phi = \frac{1+\sqrt{5}}{2} \approx 1.618$ 관계를 정밀하게 적용합니다.


In [ ]:
def get_platonic_vertices(solid_type):
    """
    선택된 플라톤 정다면체의 3차원 공간 속 꼭짓점 좌표들의 numpy 배열을 반환합니다.
    """
    # 1. 황금비(phi) 상수를 수학 공식으로 정의합니다. (대략 1.618033...)
    # 5각형의 대칭성을 지닌 입체들의 기하 좌표 생성에 필수적인 무리수 값입니다.
    phi = (1.0 + np.sqrt(5.0)) / 2.0
    
    # 2. 황금비의 역수(1 / phi = phi - 1) 상수를 정의합니다. (대략 0.618033...)
    ip = 1.0 / phi
    
    # 3. 각 다면체 타입별 꼭짓점(Vertices) 좌표 집합 리턴
    if solid_type == 'Tetrahedron (정사면체)':
        # 정사면체는 서로 마주 보는 방향으로 어긋나 있는 정육면체의 대각선 정점 4개로 형성됩니다.
        # 꼭짓점 개수 V = 4
        vertices = np.array([
            [1.0, 1.0, 1.0],      # 꼭짓점 0
            [1.0, -1.0, -1.0],    # 꼭짓점 1
            [-1.0, 1.0, -1.0],    # 꼭짓점 2
            [-1.0, -1.0, 1.0]     # 꼭짓점 3
        ])
        return vertices
        
    elif solid_type == 'Hexahedron (정육면체)':
        # 정육면체는 x, y, z 세 방향의 값이 모두 +/- 1로 조화되는 대칭 형태입니다.
        # 꼭짓점 개수 V = 8
        vertices = np.array([
            [-1.0, -1.0, -1.0],   # 꼭짓점 0
            [1.0, -1.0, -1.0],    # 꼭짓점 1
            [1.0, 1.0, -1.0],     # 꼭짓점 2
            [-1.0, 1.0, -1.0],    # 꼭짓점 3
            [-1.0, -1.0, 1.0],    # 꼭짓점 4
            [1.0, -1.0, 1.0],     # 꼭짓점 5
            [1.0, 1.0, 1.0],      # 꼭짓점 6
            [-1.0, 1.0, 1.0]      # 꼭짓점 7
        ])
        return vertices
        
    elif solid_type == 'Octahedron (정팔면체)':
        # 정팔면체는 3차원 축의 양극단 점 6개로 결합된 다이아몬드 형태입니다.
        # 꼭짓점 개수 V = 6
        vertices = np.array([
            [1.0, 0.0, 0.0],      # x축 양끝점 (0)
            [-1.0, 0.0, 0.0],     # (1)
            [0.0, 1.0, 0.0],      # y축 양끝점 (2)
            [0.0, -1.0, 0.0],     # (3)
            [0.0, 0.0, 1.0],      # z축 양끝점 (4)
            [0.0, 0.0, -1.0]      # (5)
        ])
        return vertices
        
    elif solid_type == 'Dodecahedron (정십이면체)':
        # 정십이면체는 20개의 꼭짓점을 가지고 오각형 12개로 덮여 있습니다.
        # 1) 기본 정육면체 정점 8개
        cube_verts = np.array([
            [-1.0, -1.0, -1.0], [1.0, -1.0, -1.0], [1.0, 1.0, -1.0], [-1.0, 1.0, -1.0],
            [-1.0, -1.0, 1.0], [1.0, -1.0, 1.0], [1.0, 1.0, 1.0], [-1.0, 1.0, 1.0]
        ])
        # 2) 황금비의 비율(phi)을 대수적으로 반영한 3차원 평면 위의 12개 교차점들
        # (0, +/-1/phi, +/-phi) 형태의 4개 점
        extra_a = np.array([[0.0, -ip, -phi], [0.0, ip, -phi], [0.0, -ip, phi], [0.0, ip, phi]])
        # (+/-1/phi, +/-phi, 0) 형태의 4개 점
        extra_b = np.array([[-ip, -phi, 0.0], [ip, -phi, 0.0], [-ip, phi, 0.0], [ip, phi, 0.0]])
        # (+/-phi, 0, +/-1/phi) 형태의 4개 점
        extra_c = np.array([[-phi, 0.0, -ip], [phi, 0.0, -ip], [-phi, 0.0, ip], [phi, 0.0, ip]])
        
        # np.vstack(수직 병합) 함수를 써서 총 20개의 꼭짓점 좌표 행렬을 하나로 통합합니다.
        return np.vstack([cube_verts, extra_a, extra_b, extra_c])
        
    elif solid_type == 'Icosahedron (정이십면체)':
        # 정이십면체는 12개의 꼭짓점을 가지며 20개의 합동인 정삼각형 면으로 덮여 있습니다.
        # 서로 직교하는 황금비 비율을 지닌 세 개의 엇갈린 평면 기하를 통해 정의됩니다.
        # (0, +/-1, +/-phi), (+/-1, +/-phi, 0), (+/-phi, 0, +/-1) 조합
        vertices = np.array([
            [0.0, -1.0, -phi], [0.0, 1.0, -phi], [0.0, -1.0, phi], [0.0, 1.0, phi],
            [-1.0, -phi, 0.0], [1.0, -phi, 0.0], [-1.0, phi, 0.0], [1.0, phi, 0.0],
            [-phi, 0.0, -1.0], [phi, 0.0, -1.0], [-phi, 0.0, 1.0], [phi, 0.0, 1.0]
        ])
        return vertices
        
    return np.array([])


### 2.2 볼록 껍질(Convex Hull)을 이용한 표면(F) 및 모서리(E) 추출
3차원 공간 속의 꼭짓점들만 가지고는 그래픽에 투사할 입체의 면이나 모서리를 정의할 수 없습니다.
이때, 꼭짓점들을 외곽에서 팽팽한 고무막으로 감싸는 원리인 **볼록 껍질(Convex Hull)** 알고리즘을 사용합니다.

- `ConvexHull(vertices)`을 연산하면, 껍질을 구성하는 최소 단위 면인 삼각형 면들의 리스트가 도출됩니다.
- 도출된 삼각형 면들의 꼭짓점 연결쌍을 추적하여 중복을 제거하면 3D 입체의 **유일한 모서리(Edges)** 집합을 계산해 낼 수 있습니다.


In [ ]:
def calculate_geometry_elements(vertices):
    """
    입력된 꼭짓점 좌표 배열로부터 볼록 껍질을 계산해 
    삼각형 면 리스트(triangles)와 고유 모서리 연결 정보 리스트(edges)를 추출해 반환합니다.
    """
    # 1. SciPy의 ConvexHull 알고리즘을 수행하여 볼록 다면체의 껍질을 빌드합니다.
    hull = ConvexHull(vertices)
    
    # 2. hull.simplices는 [꼭짓점A, 꼭짓점B, 꼭짓점C] 인덱스가 담긴 삼각형들의 리스트입니다. (F_calc)
    triangles = hull.simplices
    
    # 3. 중복되지 않는 고유한 모서리 세트를 찾아냅니다.
    # 파이썬의 set(집합) 구조는 동일한 항목이 중복 저장되는 것을 자동으로 막아 줍니다.
    edges = set()
    for simplex in triangles:
        # 하나의 삼각형은 3개의 모서리를 가집니다 (0->1, 1->2, 2->0).
        for i in range(3):
            p1 = simplex[i]
            p2 = simplex[(i + 1) % 3] # 마지막 인덱스일 땐 다시 처음(0)으로 돌아오게 나머지 연산(%)을 적용
            
            # 모서리의 두 끝점 인덱스를 크기순으로 정렬하여 튜플로 묶습니다. (예: (2, 5)와 (5, 2)를 동일하게 처리하기 위함)
            edge_tuple = tuple(sorted((p1, p2)))
            
            # 중복 감지 집합에 추가
            edges.add(edge_tuple)
            
    # 삼각형 면들과 고유 모서리(list로 변환)를 리턴합니다.
    return triangles, list(edges)


### 2.3 교과서 이론 수치 대조 및 대시보드 생성
수학자 오일러는 모든 볼록 다면체에 대해 꼭짓점 수($V$), 모서리 수($E$), 면 수($F$) 사이에 항상 $V - E + F = 2$가 성립함을 보였습니다. 이를 **오일러의 다면체 정리**라고 합니다.

선택된 입체의 교과서적 이론값과 3D 그래픽 연산값을 실시간 비교하는 스타일링된 HTML 대시보드 함수를 작성합니다.


In [ ]:
# 5대 정다면체의 교과서적인 완벽한 V, E, F 수치 기록 사전
theoretical_data = {
    'Tetrahedron (정사면체)': {'V': 4, 'E': 6, 'F': 4},
    'Hexahedron (정육면체)': {'V': 8, 'E': 12, 'F': 6},
    'Octahedron (정팔면체)': {'V': 6, 'E': 12, 'F': 8},
    'Dodecahedron (정십이면체)': {'V': 20, 'E': 30, 'F': 12},
    'Icosahedron (정이십면체)': {'V': 12, 'E': 30, 'F': 20}
}

def build_stats_html_panel(solid_type, V_val, E_val, F_val):
    """
    이론적 기하 수치와 실제 삼각분할 3D 모델의 기하 수치를 
    한눈에 비교하고 오일러 정리를 검증하는 예쁜 HTML 대시보드 패널 양식을 빚어냅니다.
    """
    # 교과서 이론 기준값 딕셔너리 추출
    theory = theoretical_data[solid_type]
    
    # 이론상 오일러 수치 (V - E + F) 계산 (당연히 2입니다)
    theory_euler = theory['V'] - theory['E'] + theory['F']
    
    # 실제 3D 모델(삼각 분할됨)에서 계산한 오일러 수치
    calc_euler = V_val - E_val + F_val
    
    # HTML/CSS 문자열 구조 조립
    html_template = f"""
    <div style='background: rgba(255, 255, 255, 0.8);
                backdrop-filter: blur(10px);
                border: 1.5px solid rgba(11, 121, 208, 0.2);
                box-shadow: 0 8px 32px 0 rgba(11, 121, 208, 0.15);
                border-radius: 16px; padding: 24px; max-width: 620px;
                font-family: sans-serif; line-height: 1.6; margin-bottom: 20px;'>
        <h3 style='margin: 0 0 12px 0; color: #0B79D0; border-bottom: 2.5px solid rgba(47, 168, 93, 0.35); padding-bottom: 6px;'>
            📐 {solid_type}의 기하학적 정보 및 오일러 정리 검증
        </h3>
        <table style='width: 100%; border-collapse: collapse; text-align: left;'>
            <thead>
                <tr style='border-bottom: 2px solid rgba(0,0,0,0.1);'>
                    <th style='padding: 8px 0; color: #555;'>기하 속성</th>
                    <th style='padding: 8px 0; text-align: center; color: #555;'>이론적 교과서 수치</th>
                    <th style='padding: 8px 0; text-align: center; color: #2FA85D;'>3D 그래픽스 계산 수치</th>
                </tr>
            </thead>
            <tbody>
                <tr style='border-bottom: 1px solid rgba(0,0,0,0.05);'>
                    <td style='padding: 10px 0; font-weight: bold;'>꼭짓점 개수 (V)</td>
                    <td style='text-align: center; font-size:1.1em;'>{theory['V']}</td>
                    <td style='text-align: center; font-size:1.1em; color: #2FA85D; font-weight: bold;'>{V_val}</td>
                </tr>
                <tr style='border-bottom: 1px solid rgba(0,0,0,0.05);'>
                    <td style='padding: 10px 0; font-weight: bold;'>모서리 개수 (E)</td>
                    <td style='text-align: center; font-size:1.1em;'>{theory['E']}</td>
                    <td style='text-align: center; font-size:1.1em; color: #2FA85D; font-weight: bold;'>{E_val}</td>
                </tr>
                <tr style='border-bottom: 2px solid rgba(0,0,0,0.1);'>
                    <td style='padding: 10px 0; font-weight: bold;'>면의 개수 (F)</td>
                    <td style='text-align: center; font-size:1.1em;'>{theory['F']}</td>
                    <td style='text-align: center; font-size:1.1em; color: #2FA85D; font-weight: bold;'>
                        {F_val} <span style='font-size:0.8em; color:#777; font-weight:normal;'>(삼각 폴리곤 분할)</span>
                    </td>
                </tr>
                <tr style='background: rgba(11, 121, 208, 0.05); font-size: 1.15em;'>
                    <td style='padding: 12px 10px; font-weight: bold; color: #0B79D0;'>오일러 검증 (V - E + F)</td>
                    <td style='text-align: center; font-weight: bold;'>{theory['V']} - {theory['E']} + {theory['F']} = {theory_euler}</td>
                    <td style='text-align: center; font-weight: bold; color: #2FA85D;'>{V_val} - {E_val} + {F_val} = {calc_euler}</td>
                </tr>
            </tbody>
        </table>
        <div style='margin-top: 15px; font-size: 0.85em; color: #666; font-style: italic;'>
            ※ <b>비밀 해설:</b> 컴퓨터 모니터의 3D 공간 상에 입체를 칠할 때는 다각형 면을 삼각형 조각들로 잘게 분할하여 계산(Mesh)합니다. 면(F)과 모서리(E)의 개수가 이론과 다르게 나타나더라도, 기하학적 위상 연결성은 완벽하게 똑같으므로 오일러 다면체 상수 <b>2</b>는 소름 돋을 정도로 완벽히 지켜집니다.
        </div>
    </div>
    """
    return html_template


### 2.4 Plotly 3D 인터랙티브 그래픽 구성
꼭짓점 점 마커, 뼈대를 그려줄 모서리 선분, 그리고 면을 칠해줄 반투명 3D 폴리곤 트레이스를 결합하여 인터랙티브 3D 플롯을 구현하는 메인 렌더링 함수를 작성합니다.


In [ ]:
def plot_platonic_solid_simulation(solid_type):
    """
    선택된 입체의 기하 계산 결과를 기반으로 Plotly 3D 시각화를 실시간 구동합니다.
    """
    # 1. 꼭짓점 데이터 추출
    vertices = get_platonic_vertices(solid_type)
    if len(vertices) == 0:
        print("기하 데이터를 불러오지 못했습니다.")
        return
        
    # 2. 기하 요소 연산 (면 및 고유 모서리 인덱스 추출)
    triangles, edges = calculate_geometry_elements(vertices)
    
    # 3. 주피터 상단에 HTML 통계 리포트 패널을 먼저 노출합니다.
    dashboard_view = build_stats_html_panel(solid_type, len(vertices), len(edges), len(triangles))
    display(HTML(dashboard_view))
    
    # 4. 꼭짓점 (Point Markers) 3D 트레이스 정의 (파란색 구슬 형태)
    trace_vertices = go.Scatter3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        mode='markers', # 마커 형태로 표현
        marker=dict(
            size=7,
            color='#0B79D0', # 브랜드 블루
            opacity=0.9,     # 약간의 투명도
            line=dict(color='white', width=2) # 흰색 원형 테두리선 부착
        ),
        name='꼭짓점 (V)'
    )
    
    # 5. 모서리 (Lines) 3D 트레이스 정의 (초록색 굵은 뼈대 선)
    # 3D 라인을 빠르게 그리기 위해 꼭짓점A 좌표, 꼭짓점B 좌표를 넣고 중간에 None을 끼워 끊어 주는 방식을 씁니다.
    line_x, line_y, line_z = [], [], []
    for edge in edges:
        pt1 = vertices[edge[0]]
        pt2 = vertices[edge[1]]
        
        # 선분의 두 끝 좌표 입력 후 끊기 기호인 None 추가
        line_x.extend([pt1[0], pt2[0], None])
        line_y.extend([pt1[1], pt2[1], None])
        line_z.extend([pt1[2], pt2[2], None])
        
    trace_edges = go.Scatter3d(
        x=line_x, y=line_y, z=line_z,
        mode='lines', # 오직 선분으로만 연결
        line=dict(
            color='#2FA85D', # 브랜드 그린
            width=5         # 뼈대 굵기
        ),
        name='모서리 (E)',
        hoverinfo='skip' # 모서리 위 마우스 호버 비활성화
    )
    
    # 6. 면 (Mesh Surfaces) 3D 트레이스 정의 (반투명 청록 글래스모피즘)
    trace_faces = go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=triangles[:, 0], j=triangles[:, 1], k=triangles[:, 2], # 삼각형 세 정점 인덱스 대응
        color='#13a89e',   # 은은한 청록색 컬러
        opacity=0.32,      # 입체도형의 뒷면 모서리와 뼈대가 비쳐 보이도록 32%의 투명도 부여
        flatshading=True,  # 평평한 쉐이딩 처리로 조명에 따른 면의 기하 각도를 입체적으로 드러냄
        name='표면 (F)',
        showlegend=True
    )
    
    # 7. 레이아웃 조율 (어두운 느낌의 고급스러운 그리드 레이아웃)
    layout = go.Layout(
        title=dict(
            text=f'<b>{solid_type} 3D 인터랙티브 탐색기</b><br><span style="font-size:12px;color:#666;">마우스 드래그로 면 회전 | 마우스 휠로 크기 확대/축소 가능</span>',
            x=0.5, # 차트 제목 가운데 정렬
            font=dict(size=18, color='#1F2937')
        ),
        scene=dict(
            # x, y, z 각 축의 가이드 면 색상과 화이트 그리드 설정
            xaxis=dict(backgroundcolor="rgb(245, 247, 250)", gridcolor="white", showbackground=True, zerolinecolor="white", range=[-2.2, 2.2]),
            yaxis=dict(backgroundcolor="rgb(245, 247, 250)", gridcolor="white", showbackground=True, zerolinecolor="white", range=[-2.2, 2.2]),
            zaxis=dict(backgroundcolor="rgb(245, 247, 250)", gridcolor="white", showbackground=True, zerolinecolor="white", range=[-2.2, 2.2]),
            aspectratio=dict(x=1, y=1, z=1) # 기하 왜곡이 생기지 않는 등배율 비율 보장
        ),
        margin=dict(l=0, r=0, b=0, t=65),
        width=720, height=600,
        showlegend=True,
        legend=dict(x=0.8, y=0.9)
    )
    
    # 최종 피규어를 빌드하고 주피터 상에 플로팅합니다.
    fig = go.Figure(data=[trace_faces, trace_edges, trace_vertices], layout=layout)
    fig.show()


### 2.5 시뮬레이터 인터랙티브 위젯 작동
드롭다운 위젯과 시뮬레이션 렌더링 함수를 연동하여 사용자의 마우스 클릭에 따라 즉각적인 다면체 변화를 보여주는 대시보드를 최종 기동합니다.


In [ ]:
# widgets.Dropdown 위젯을 생성하여 사용자가 플라톤 다면체 리스트를 고르게 합니다.
solid_dropdown_widget = widgets.Dropdown(
    options=[
        'Tetrahedron (정사면체)', 
        'Hexahedron (정육면체)', 
        'Octahedron (정팔면체)', 
        'Dodecahedron (정십이면체)', 
        'Icosahedron (정이십면체)'
    ],
    value='Tetrahedron (정사면체)', # 최초 실행 시 디폴트 선택값
    description='정다면체 선택:',
    style={'description_width': 'initial'} # 설명 문구 너비의 잘림 방지 스타일 적용
)

# widgets.interact 함수를 호출하면, 드롭다운 위젯의 선택이 바뀔 때마다 파라미터로 바인딩된 draw 함수가 자동 재실행됩니다.
widgets.interact(plot_platonic_solid_simulation, solid_type=solid_dropdown_widget);


---
## 3. 실습 2: 2D 단면에서 3D 회전체 생성 가상 모델러
평면 위의 한 도형(단면)을 고정된 일직선(회전축)을 중심으로 $360^\circ$ 회전시켰을 때 그려지는 3차원 입체를 **회전체**라고 합니다.

이 가상 모델러 실습에서는 회전각의 변화(0도부터 360도까지)를 슬라이더 위젯으로 제어하면서, 2D 단면이 3차원 허공에 흔적을 남기며 부피를 가지는 3D 스킨으로 채워지는 실시간 변화 과정을 시각적으로 확인합니다.

### 3.1 2D 단면 프로파일(Profile) 곡선의 정의
회전체의 뼈대가 될 $xy$평면(z=0) 상의 2D 점 좌표들($x$ 반경 배열, $y$ 높이 배열)을 리턴하는 수학적 계산 함수를 선언합니다.


In [ ]:
def get_2d_profile_points(profile_type, r_scale, h_scale, steps=40):
    """
    선택한 입체에 부합하는 xy 평면 상의 2차원 외곽 윤곽선 좌표(x, y)를 계산하여 리턴합니다.
    x 좌표는 회전축(y축)으로부터의 거리를 나타내며 반드시 0 이상(x >= 0)이어야 합니다.
    """
    # 0부터 1까지 균등하게 나뉜 파라미터 간격 배열 생성
    t = np.linspace(0, 1, steps)
    
    if profile_type == 'Cylinder (원기둥)':
        # 직사각형을 회전시키면 원기둥이 만들어집니다.
        # 외각 윤곽을 만들기 위해 (0, 바닥) -> (r, 바닥) -> (r, 뚜껑) -> (0, 뚜껑)의 4개 꼭짓점으로 된 단면을 빚습니다.
        x_coords = np.array([0.0, r_scale, r_scale, 0.0])
        y_coords = np.array([-h_scale/2.0, -h_scale/2.0, h_scale/2.0, h_scale/2.0])
        
    elif profile_type == 'Cone (원뿔)':
        # 직각삼각형을 한 변(높이축)을 중심으로 돌리면 원뿔이 탄생합니다.
        # (0, 바닥) -> (r, 바닥) -> (0, 꼭대기)의 3개 제어점으로 닫힌 삼각형을 만듭니다.
        x_coords = np.array([0.0, r_scale, 0.0])
        y_coords = np.array([-h_scale/2.0, -h_scale/2.0, h_scale/2.0])
        
    elif profile_type == 'Sphere (구)':
        # 반원을 지름을 축으로 회전시키면 구가 탄생합니다.
        # 각도 범위를 -90도(-pi/2)부터 +90도(pi/2)로 나누어 x=cos(각도), y=sin(각도) 원형 반원 궤적을 그립니다.
        semi_angles = np.linspace(-np.pi/2.0, np.pi/2.0, steps)
        x_coords = r_scale * np.cos(semi_angles)
        y_coords = (h_scale / 2.0) * np.sin(semi_angles)
        
        # 입체가 닫힌 형태를 갖추도록 처음과 끝에 중심축 x=0의 정점(바닥, 뚜껑)을 덧대어 합칩니다(np.concatenate).
        x_coords = np.concatenate([[0.0], x_coords, [0.0]])
        y_coords = np.concatenate([[-h_scale/2.0], y_coords, [h_scale/2.0]])
        
    elif profile_type == 'Frustum (원뿔대)':
        # 사다리꼴을 돌리면 원뿔대가 됩니다.
        # 바닥 반경은 r_scale, 상단 뚜껑 반경은 40% 크기인 r_scale * 0.4로 정량합니다.
        x_coords = np.array([0.0, r_scale, r_scale * 0.4, 0.0])
        y_coords = np.array([-h_scale/2.0, -h_scale/2.0, h_scale/2.0, h_scale/2.0])
        
    elif profile_type == 'Torus (도넛)':
        # 축에서 멀찍이 떨어진 완전한 원 단면을 회전하면 튜브 도넛 형태인 토러스가 생성됩니다.
        # 원의 중심을 x=r_scale 만큼 우측으로 오프셋 밀고, 단면 원 반경(두께)은 r_scale * 0.35로 정의합니다.
        r_thickness = r_scale * 0.35
        circle_angles = np.linspace(0, 2*np.pi, steps)
        x_coords = r_scale + r_thickness * np.cos(circle_angles) # 우측 평행이동 반영
        y_coords = r_thickness * np.sin(circle_angles)
        
    else: # Custom Vase (호리병)
        # 삼각함수를 결합해 물결치며 굴곡이 생기는 예술적인 곡선을 구성합니다.
        x_coords = r_scale * (0.6 + 0.3 * np.sin(t * np.pi * 2.5))
        y_coords = h_scale * (t - 0.5)
        # 입체 밀폐용 시작/끝 정점 결합
        x_coords = np.concatenate([[0.0], x_coords, [0.0]])
        y_coords = np.concatenate([[-h_scale/2.0], y_coords, [h_scale/2.0]])
        
    return x_coords, y_coords


### 3.2 3D 공간 상의 회전 궤적 격자(Mesh Grid) 연산
2D 단면 곡선의 각 정점 $(x_i, y_i)$이 $y$축을 중심으로 회전 각도 $\phi$가 $0$에서 $\theta_{max}$까지 쓸고 지나간 흔적의 3차원 격자 행렬 $(X, Y, Z)$을 연산합니다.

수학적 회전 변환 공식:
$$X = x_i \cos\phi$$
$$Z = x_i \sin\phi$$
$$Y = y_i$$

이 공식은 Numpy의 **외적(np.outer)** 연산을 사용해 모든 각도 세트와 꼭짓점 세트의 조합을 한 번에 연산하여 성능을 극대화합니다.


In [ ]:
def calculate_revolution_mesh_grid(x_line, y_line, max_deg, steps_angle=60):
    """
    2차원 윤곽 좌표를 입력받아 y축 회전에 따른 3D 표면 메시의 X, Y, Z 좌표 매트릭스를 반환합니다.
    """
    # 1. 사용자가 슬라이더로 조절한 각도(Degree) 값을 파이썬 삼각함수 연산용 라디안(Radian)으로 변환합니다.
    max_rad = np.radians(max_deg)
    
    # 2. 0부터 최대 라디안 범위까지 각도 스텝을 생성합니다.
    phi_steps = np.linspace(0, max_rad, steps_angle)
    
    # 3. np.outer(외적) 함수를 이용해 회전 3D 스킨의 격자 좌표들을 순식간에 계산합니다.
    # X 격자 매트릭스: 각 각도 성분(cos)과 단면의 반경거리(x_line) 곱의 모든 조합 조합 생성
    X_matrix = np.outer(np.cos(phi_steps), x_line)
    
    # Z 격자 매트릭스: 각 각도 성분(sin)과 단면의 반경거리(x_line) 곱의 조합 생성
    Z_matrix = np.outer(np.sin(phi_steps), x_line)
    
    # Y 격자 매트릭스: 높이는 회전에 영향받지 않으므로 단면의 높이(y_line) 성분을 그대로 복사 채움
    Y_matrix = np.outer(np.ones_like(phi_steps), y_line)
    
    return X_matrix, Y_matrix, Z_matrix


### 3.3 회전체 3D 렌더링 시각화 엔진
이제 연산된 격자 배열을 이용해 `go.Surface` 회전 스킨을 그리고, **원래의 빨간색 2D 단면**, **회전 경계에 위치한 주황색 2D 단면**, 그리고 점선으로 된 **중앙 회전축(y축)**을 오버레이하여 입체적으로 연출하는 렌더러 함수를 작성합니다.


In [ ]:
def render_revolution_solid_scene(profile_type, max_angle, r_scale, h_scale):
    """
    슬라이더로 조작된 파라미터 수치들에 맞춰 3D 공간 상에 회전체를 실시간 가시화합니다.
    """
    # 1. 2D 단면 곡선 x, y 데이터 생성
    x_profile, y_profile = get_2d_profile_points(profile_type, r_scale, h_scale)
    
    # 2. 3D 회전 격자 데이터 연산
    X_grid, Y_grid, Z_grid = calculate_revolution_mesh_grid(x_profile, y_profile, max_angle)
    
    # 3. 3D 캔버스에 추가할 그래픽 트레이스들의 리스트
    traces_list = []
    
    # 4. 회전 각도가 조금이라도 돌아갔을 때(>0)만 입체 서페이스 표면을 렌더링합니다.
    if max_angle > 0:
        # go.Surface는 X, Y, Z 격자를 바탕으로 표면에 연속적인 컬러를 입혀서 3차원 입체로 렌더링해 줍니다.
        trace_surface = go.Surface(
            x=X_grid, y=Y_grid, z=Z_grid,
            # 아름다운 브랜드 그라데이션 컬러스케일 적용 (Blue #0B79D0 -> Green #2FA85D)
            colorscale=[[0.0, '#0B79D0'], [1.0, '#2FA85D']],
            opacity=0.75,       # 내부 회전 중심축이 반투명하게 투과되어 보이도록 불투명도 75% 설정
            showscale=False,   # 컬러 범위 막대는 숨김 처리하여 화면 최적화
            name='회전체 표면 스킨'
        )
        traces_list.append(trace_surface)
        
    # 5. 회전 출발선인 '원래 2D 단면'(z = 0 평면상에 고정)을 빨간색 라인으로 겹쳐 그립니다.
    trace_original_profile = go.Scatter3d(
        x=x_profile, y=y_profile, z=np.zeros_like(x_profile),
        mode='lines+markers',
        line=dict(color='#E11D48', width=6), # 강한 장미빛 레드 계열 선
        marker=dict(size=4, color='#E11D48'),
        name='2D 오리지널 단면(시작점)'
    )
    traces_list.append(trace_original_profile)
    
    # 6. 회전이 360도 꽉 찬 상태가 아닐 때(0도 ~ 360도 미만)는 회전 각도 종단점의 단면 경계선을 
    # 주황색 점선으로 함께 렌더링하여 역동적으로 쓸고 나가는 경계를 알려줍니다.
    if 0 < max_angle < 360:
        rad = np.radians(max_angle)
        # 종단점 각도에 맞춰 단면 좌표 3D 회전 변환 적용
        curr_x = x_profile * np.cos(rad)
        curr_z = x_profile * np.sin(rad)
        
        trace_rotating_profile = go.Scatter3d(
            x=curr_x, y=y_profile, z=curr_z,
            mode='lines',
            line=dict(color='#F59E0B', width=5, dash='dash'), # 노란 점선
            name='현재 회전 경계선'
        )
        traces_list.append(trace_rotating_profile)
        
    # 7. 기하학적 핵심 회전축인 y축을 한가운데 기둥 점선으로 구성합니다.
    axis_y_coords = np.linspace(-h_scale * 0.8, h_scale * 0.8, 10)
    trace_axis = go.Scatter3d(
        x=np.zeros_like(axis_y_coords), y=axis_y_coords, z=np.zeros_like(axis_y_coords),
        mode='lines',
        line=dict(color='#374151', width=3, dash='dot'), # 어두운 회색 점선
        name='회전축 (y축)'
    )
    traces_list.append(trace_axis)
    
    # 8. 3D 입체 투영 렌더링 경계 범위 설정
    space_bound = max(r_scale, h_scale / 2.0) * 1.4
    
    # 9. 레이아웃 파라미터 조립
    layout = go.Layout(
        title=dict(
            text=f'<b>{profile_type} 3D 가상 회전체 생성 (회전각: {max_angle}°)</b>',
            x=0.5, # 가운데 정렬
            font=dict(size=18, color='#1F2937')
        ),
        scene=dict(
            xaxis=dict(title='X (반경 방향)', range=[-space_bound, space_bound], backgroundcolor="rgb(248, 250, 252)", showbackground=True),
            yaxis=dict(title='Y (회전축 방향)', range=[-space_bound, space_bound], backgroundcolor="rgb(248, 250, 252)", showbackground=True),
            zaxis=dict(title='Z (깊이 방향)', range=[-space_bound, space_bound], backgroundcolor="rgb(248, 250, 252)", showbackground=True),
            aspectratio=dict(x=1, y=1, z=1) # 1:1:1 종횡비 설정
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        width=750, height=600,
        showlegend=True,
        legend=dict(x=0.02, y=0.98)
    )
    
    # 3D 피규어를 생성하고 출력합니다.
    fig = go.Figure(data=traces_list, layout=layout)
    fig.show()


### 3.4 대시보드 위젯 조립 및 회전 시뮬레이션 작동
사용자가 조절할 수 있는 4개의 위젯(단면 유형 드롭다운, 회전각 슬라이더, 반지름 슬라이더, 높이 슬라이더)을 가로/세로 레이아웃으로 묶어 배치하여 모델러를 가동합니다.


In [ ]:
# 1. 단면 종류를 선택하기 위한 드롭다운 메뉴 위젯 정의
input_profile_dropdown = widgets.Dropdown(
    options=['Cylinder (원기둥)', 'Cone (원뿔)', 'Sphere (구)', 'Frustum (원뿔대)', 'Torus (도넛)', 'Vase (호리병)'],
    value='Cone (원뿔)',
    description='1. 단면 형태:',
    style={'description_width': 'initial'}
)

# 2. 회전 각도를 제어할 슬라이더 정의 (0도에서 360도까지 5도 단위)
input_angle_slider = widgets.IntSlider(
    value=180,
    min=0, max=360, step=5,
    description='2. 회전 각도 (deg):',
    style={'description_width': 'initial'},
    continuous_update=True # 슬라이더 바를 끄는 도중에도 3D 서페이스가 끊김 없이 실시간 업데이트됨
)

# 3. 단면의 반지름 크기를 조절하는 슬라이더 정의
input_radius_slider = widgets.FloatSlider(
    value=1.5,
    min=0.5, max=3.0, step=0.1,
    description='4. 단면 폭 (Radius):',
    style={'description_width': 'initial'}
)

# 4. 입체 기둥의 높이를 조절하는 슬라이더 정의
input_height_slider = widgets.FloatSlider(
    value=3.0,
    min=1.0, max=5.0, step=0.1,
    description='5. 입체 높이 (Height):',
    style={'description_width': 'initial'}
)

# 5. 제어 컨트롤 박스를 세로형 VBox 컨테이너에 깔끔하게 배치합니다.
dashboard_control_box = widgets.VBox([
    widgets.HTML("<h4 style='color:#0B79D0; margin:0 0 10px 0;'>🔧 실시간 매개변수 제어창</h4>"),
    input_profile_dropdown,
    input_angle_slider,
    input_radius_slider,
    input_height_slider
])

# 6. widgets.interactive_output 함수를 통해 위젯 입력값들을 시각화 렌더러 함수의 인자 값들과 1:1로 매핑 결합합니다.
rendered_output = widgets.interactive_output(
    render_revolution_solid_scene,
    {
        'profile_type': input_profile_dropdown,
        'max_angle': input_angle_slider,
        'r_scale': input_radius_slider,
        'h_scale': input_height_slider
    }
)

# 7. 왼쪽(조작 패널)과 오른쪽(3D 그래픽 창)을 가로형 HBox 컨테이너로 조립하여 최종 화면에 디스플레이합니다.
display(widgets.HBox([dashboard_control_box, rendered_output]))


---
## 4. 깊이 있는 기하학 사색을 위한 질문

1. **다면체의 오일러 정리($V - E + F = 2$)와 위상 수학**:
   우리가 다룬 5가지 플라톤 정다면체는 내부에 구멍이 없는 완벽한 구(Sphere)와 위상학적으로 완벽히 똑같은(Homeomorphic) 구조를 가집니다. 따라서 오일러 상수는 항상 **2**가 나옵니다.
   그렇다면, 만약 가운데에 도넛과 같이 구멍이 1개 뚫린 입체 다면체(Torus 형태)를 만들고 그 다면체의 꼭짓점(V), 모서리(E), 면(F)의 개수를 조밀하게 세어 $V - E + F$ 공식을 계산한다면, 그 결과 상수는 어떤 정수 값으로 변하게 될까요? (가상 회전체로 구현한 '도넛(Torus)'을 보면서 상상력을 발휘해 보세요.)

2. **2D 윤곽선과 3D 부피 사이의 다리**:
   2차원 상의 가느다란 점과 선들을 한 축을 중심으로 연속적으로 돌리는 단순한 연산만으로 입체적인 공간의 부피를 창조했습니다.
   우리의 삶이나 비즈니스 비전 속에서 가치 있는 결과물(3차원의 부피를 가진 입체)을 창조하기 위해, 우리가 매일 끊임없이 축을 잡고 회전시키며 반복하고 있는 나만의 '회전축'과 '2차원 단면 곡선'은 각각 무엇에 비유될 수 있을까요? 깊이 있게 사유해 보세요.
